In [2]:
# Import library require for process
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
import warnings as ws
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
ws.filterwarnings('ignore')



# RFE - input -> indep_X and output  dep_Y and n - K times
# chi2 = for evalution method
# this is feature selection before model creation
# Once selct the feature, return back in this function
def rfe_feature_selection(indep_X,dep_Y,n):
        rfelist = []    
        regressorLinear = LinearRegression()
        regressorRidge = Ridge()
        regressorLasso = Lasso(alpha=0.1)
        regressorDeci = DecisionTreeRegressor(random_state = 0)
        regressorSVR = SVR(kernel='linear')
        regressorRandom = RandomForestRegressor(n_estimators=100, random_state=42)
        
        rfemodellist = [regressorLinear, regressorRidge, regressorLasso, regressorDeci, regressorSVR, regressorRandom]
        
        for i in rfemodellist:
            log_rfe = RFE(estimator=i, n_features_to_select=n)
            log_fit = log_rfe.fit(indep_X, dep_Y)
            log_rfe_feature = log_fit.transform(indep_X)
            rfelist.append(log_rfe_feature)        
        return rfelist
    
#split_scalar - Split the input, output train and test set. then changes the input to scalar value    
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test

# r2_prediction - used for regression method, model prediction evaluate method
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
    
# Linear method is used for Linear regression model creation and r2 prediction
def fit_linear(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LinearRegression
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   

# Ridge method is used for svm model creation and r2 prediction
def fit_ridge(X_train,y_train,X_test):                
        from sklearn.linear_model import Ridge
        regressor = Ridge()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
    
# Lasso method is used for svm_NL model creation and r2 prediction   
def fit_lasso(X_train,y_train,X_test):                
        from sklearn.linear_model import Lasso
        regressor = Lasso(alpha=0.1)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# Decision method is used for Decision tree model creation and r2 prediction   
def fit_decision(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training setC
        from sklearn.tree import DecisionTreeRegressor
        regressor = DecisionTreeRegressor(random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# SVR method is used for random forest model creation and r2 prediction  
def fit_svr(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.svm import SVR
        regressor = SVR(kernel='linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
# random method is used for random forest model creation and r2 prediction  
def fit_random(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestRegressor
        regressor = RandomForestRegressor(n_estimators=100, random_state=42)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
    
# RFE_regression method is used for create dataset with columns name as 'Linear','SVMl','SVMnl','Decision','Random' and index ChiSquare
# and fill the each columns values 
def RFE_regression(acclin,accsRid,accLa,accdes,accrSvR,accrf): 
    
    dataframe=pd.DataFrame(index=['Linear', 'Ridge', 'Lasso', 'Decision', 'SVR', 'Random'],columns=['Linear','Ridge','Lasso','Decision', 'SVR', 'Random'
                                                                                     ])

    for number,idex in enumerate(dataframe.index):
        
        dataframe['Linear'][idex]=acclin[number]       
        dataframe['Ridge'][idex]=accsRid[number]
        dataframe['Lasso'][idex]=accLa[number]
        dataframe['Decision'][idex]=accdes[number]
        dataframe['SVR'][idex]=accrSvR[number]
        dataframe['Random'][idex]=accrf[number]
    return dataframe
    

In [6]:
# Read data from file and datatset should without index
dataset=pd.read_csv("prep.csv",index_col=None)
df2=dataset
# Preprocessed by one hot encoding
df2 = pd.get_dummies(df2, drop_first=True)
# assign the input only
indep_X=df2.drop('classification_yes', axis=1)
# assign output only
dep_Y=df2['classification_yes']

# choose the feature selection here using n feature 
RFEList=rfe_feature_selection(indep_X,dep_Y,8)      


In [7]:
# Create 5 empty list for each algorithm and split the input and output
# Evalute each algorithmwise r2 score and send RFE_regression funtion
# finally the evalution data represent by table view.
acclin=[]
accsRid=[]
accLa=[]
accdes=[]
accrf=[]
accrSvR=[]
accrf =[]


for i in RFEList:   
    X_train, X_test, y_train, y_test=split_scalar(i,dep_Y)  
    r2_lin=fit_linear(X_train,y_train,X_test)
    acclin.append(r2_lin)
    
    r2_ri=fit_ridge(X_train,y_train,X_test)    
    accsRid.append(r2_ri)
    
    r2_la=fit_lasso(X_train,y_train,X_test)
    accLa.append(r2_la)
    
    r2_d=fit_decision(X_train,y_train,X_test)
    accdes.append(r2_d)
    
    r2_sr=fit_svr(X_train,y_train,X_test)
    accrSvR.append(r2_sr)

    r2_r=fit_random(X_train,y_train,X_test)
    accrf.append(r2_r)
    
    
result=RFE_regression(acclin,accsRid,accLa,accdes,accrSvR,accrf)

In [5]:
result
# 9

,Linear,Ridge,Lasso,Decision,SVR,Random
Linear,0.716216,0.71625,0.599767,0.968654,0.684977,0.962849
Ridge,0.716216,0.71625,0.599767,0.968654,0.684977,0.962849
Lasso,0.60317,0.603203,0.538829,0.869792,0.564522,0.835694
Decision,0.702878,0.703025,0.579447,0.826389,0.672776,0.92783
SVR,0.700809,0.700773,0.599767,0.82978,0.682756,0.927438
Random,0.715774,0.715834,0.581397,0.782986,0.681605,0.912287


In [8]:
result
# 8

,Linear,Ridge,Lasso,Decision,SVR,Random
Linear,0.709204,0.709165,0.599767,0.952168,0.684292,0.939159
Ridge,0.709204,0.709165,0.599767,0.952168,0.684292,0.939159
Lasso,0.604073,0.604097,0.538829,0.826389,0.562445,0.837778
Decision,0.703917,0.704033,0.579447,0.782986,0.673437,0.931055
SVR,0.701052,0.701088,0.599767,0.82978,0.679964,0.930577
Random,0.701652,0.701497,0.573936,0.782986,0.679374,0.915178
